In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords

# Ensure stopwords are downloaded
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def load_dataset(filepath):
    """Loads IMDb dataset, analyzes missing values, and removes incomplete rows."""
    df = pd.read_csv(filepath)

    # Store original dataset size
    original_size = df.shape[0]

    # Check missing values
    missing_counts = df[['Series_Title', 'Overview', 'Genre']].isna().sum()
    print("Missing Values Before Cleaning:\n", missing_counts)

    # Drop rows with missing values in any of these columns
    df = df.dropna(subset=['Series_Title', 'Overview', 'Genre'])

    # Calculate number of removed rows
    removed_rows = original_size - df.shape[0]

    # Print dataset size after cleaning
    print(f"Dataset size before cleaning: {original_size} movies")
    print(f"Dataset size after cleaning: {df.shape[0]} movies")
    print(f"Total rows removed: {removed_rows}")

    return df[['Series_Title', 'Overview', 'Genre']]

def preprocess_text(text):
    """Converts text to lowercase and removes stopwords."""
    text = text.lower()
    text = " ".join(word for word in text.split() if word not in stop_words)
    return text

def preprocess_data(df):
    """Applies text preprocessing to Overview and Genre columns."""
    df['Overview'] = df['Overview'].astype(str).apply(preprocess_text)
    df['Genre'] = df['Genre'].astype(str).apply(lambda x: x.lower())  # Convert genres to lowercase
    return df

def extract_genres_from_input(user_input, unique_genres):
    """Extracts genre-related keywords from user input based on unique genres from the dataset."""
    extracted_genres = [genre for genre in unique_genres if genre in user_input.lower()]
    return ", ".join(extracted_genres) if extracted_genres else None

def genre_similarity(user_genre, movie_genre):
    """Calculates similarity between user genre and movie genre."""
    user_genres = set(user_genre.split(','))
    movie_genres = set(movie_genre.split(','))
    return len(user_genres.intersection(movie_genres))

def compute_similarity(user_input, user_genre, overviews, genres):
    """Computes combined similarity between user input and movie overviews and genres."""
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform([user_input] + overviews.tolist())
    cosine_sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:])

    # Calculate genre similarity for each movie
    genre_similarities = [genre_similarity(user_genre, genre) for genre in genres]

    # Normalize cosine similarity
    max_cosine_sim = max(cosine_sim[0]) if max(cosine_sim[0]) > 0 else 1
    normalized_cosine_sim = [sim / max_cosine_sim for sim in cosine_sim[0]]

    # Weighting: Give more weight to genres if present, otherwise rely on overview
    if user_genre:
        combined_similarity = [
            0.7 * genre_sim + 0.3 * overview_sim
            for genre_sim, overview_sim in zip(genre_similarities, normalized_cosine_sim)
        ]
    else:
        combined_similarity = normalized_cosine_sim  # Use only overview similarity if no genres match

    return combined_similarity

def recommend(user_input, df, top_n=5):
    """Returns top N recommended movies based on combined genre and overview similarity."""
    df = preprocess_data(df)

    # Extract unique genres from dataset
    unique_genres = set()
    for genre_list in df['Genre']:
        unique_genres.update(genre_list.split(', '))

    # Extract genres from user input
    user_genre = extract_genres_from_input(user_input, unique_genres)

    if not user_genre:
        print("\n No genre detected in user input. Using overview-based recommendations only.\n")
        user_genre = ""

    user_input = preprocess_text(user_input)  # Preprocess user input
    combined_similarity_scores = compute_similarity(user_input, user_genre, df['Overview'], df['Genre'])

    df['similarity'] = combined_similarity_scores

    # Apply a similarity threshold (e.g., discard movies with similarity < 0.5)
    df = df[df['similarity'] > 0.5]

    if df.empty:
        print("\n No highly similar movies found. Showing top-rated fallback recommendations.\n")
        return df.nlargest(top_n, 'Series_Title')  # Fallback: Recommend highest-rated movies instead

    return df.nlargest(top_n, 'similarity')[['Series_Title', 'similarity']]

if __name__ == "__main__":
    dataset_path = "imdb_top_1000.csv"
    user_query = input("Describe the type of movie you like: ")

    data = load_dataset(dataset_path)
    recommendations = recommend(user_query, data)

    print("\n Top Movie Recommendations:\n", recommendations)
